## 1. Setup

Installs, imports, load .env

In [ ]:
%pip install -q -r ../requirements.txt

import json
import os
import sys
from pathlib import Path

sys.path.append("../src")

import pandas as pd

print("Setup complete ✓")

## 2. Config

Model names, API base URLs, temperature, max_tokens, fixed label list

In [ ]:
from config import (
    DATA_PATH, RESULTS_DIR, LABELS, MODELS, API_PRICING_PER_1M,
    RUN_DATE, TEMPERATURE, MAX_TOKENS, OPENAI_API_KEY,
)

print("LABELS:", LABELS)
print("MODELS:", MODELS)
print("TEMPERATURE:", TEMPERATURE, " MAX_TOKENS:", MAX_TOKENS)
print("RUN_DATE:", RUN_DATE)
print("OPENAI_API_KEY loaded:", "✓" if OPENAI_API_KEY else "✗ — add it to .env")

## 3. Prompt template

In [ ]:
from prompt import build_prompt

# sanity check: see the exact frozen prompt every model receives
print(build_prompt('print("hello world")'))

## 4. Load data

In [ ]:
from collections import Counter
from run import load_items

ITEMS = load_items()

# Validate — fail loudly; never silently skip bad labels
invalid = [item["id"] for item in ITEMS if item["expected"] not in LABELS]
if invalid:
    raise ValueError(f"Items with unknown labels (fix before running): ids={invalid}")

print(f"Total items: {len(ITEMS)}")
print("\nLabel distribution:")
for lang, count in sorted(Counter(item["expected"] for item in ITEMS).items()):
    print(f"  {lang:<12} {count}")

## 5. Run: Top API model

In [ ]:
from run import run_model, write_csv

PER_ITEM_CSV = RESULTS_DIR / "per_item.csv"

# Calls the real Groq API for all 50 items and appends to results/per_item.csv.
# Re-running this cell re-calls the API and appends duplicate rows — delete
# per_item.csv first if you want a clean re-run.
top_rows = run_model("top", ITEMS)
write_csv(top_rows, path=PER_ITEM_CSV, append=True)

## 6. Run: Cheap API model

In [ ]:
cheap_rows = run_model("cheap", ITEMS)
write_csv(cheap_rows, path=PER_ITEM_CSV, append=True)

## 7. Run: Open-weights model

In [ ]:
# Requires a running vLLM server at MODELS["local"]["base_url"], and a real
# model name filled into config.py's MODELS["local"]["model"].
# Uncomment once your teammate's vLLM server is up:

# local_rows = run_model("local", ITEMS)
# write_csv(local_rows, path=PER_ITEM_CSV, append=True)

print("Local model: pending — waiting on vLLM setup (see config.py MODELS['local'])")

## 8. Score

In [ ]:
# run.py already scores each row at call time using score.py's exact-match
# logic, so this just loads the accumulated results for analysis.
per_item = pd.read_csv(PER_ITEM_CSV)
print(f"Loaded {len(per_item)} rows across models: {sorted(per_item['model_key'].unique())}")
per_item.head()

## 9. Cost

In [ ]:
from cost import ApiPricing, api_cost_per_1k_requests, cost_at_volume

cost_rows = []
for model_key in ["top", "cheap"]:
    sub = per_item[per_item["model_key"] == model_key]
    if sub.empty:
        continue
    avg_in = sub["input_tokens"].mean()
    avg_out = sub["output_tokens"].mean()

    pricing = ApiPricing(
        price_per_1k_input=API_PRICING_PER_1M[model_key]["input"] / 1000,
        price_per_1k_output=API_PRICING_PER_1M[model_key]["output"] / 1000,
    )
    cost_per_1k = api_cost_per_1k_requests(avg_in, avg_out, pricing)

    cost_rows.append({
        "model_key": model_key,
        "avg_input_tokens": round(avg_in, 1),
        "avg_output_tokens": round(avg_out, 1),
        "cost_per_1k_requests_usd": round(cost_per_1k, 4),
        "cost_at_100x_traffic_usd": round(cost_at_volume(cost_per_1k / 1000, 100_000), 2),
    })

# Local model cost uses hardware $/hour instead of token pricing — add once
# hardware_note.md has real numbers, e.g.:
# from cost import local_cost_per_1k_requests
# local_cost_1k = local_cost_per_1k_requests(hardware_cost_per_hour=..., requests_per_hour=...)

cost_df = pd.DataFrame(cost_rows)
cost_df

## 10. Results summary table

In [ ]:
summary_rows = []
for model_key, group in per_item.groupby("model_key"):
    n_correct = group["correct"].sum()
    n_total = len(group)
    summary_rows.append({
        "model_key": model_key,
        "accuracy": f"{n_correct}/{n_total}",
        "accuracy_pct": round(100 * n_correct / n_total, 1),
        "p50_latency_ms": round(group["latency_ms"].quantile(0.50), 1),
        "p95_latency_ms": round(group["latency_ms"].quantile(0.95), 1),
        "parse_errors": (group["status"] == "parse_error").sum(),
        "timeouts": (group["status"] == "timeout").sum(),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / "summary.csv", index=False)
summary_df

## 11. Sample wrong answers

In [ ]:
for model_key, group in per_item.groupby("model_key"):
    wrong = group[group["correct"] == False].head(3)
    print(f"\n=== {model_key}: wrong/error answers ===")
    for _, row in wrong.iterrows():
        print(f"  item {row['item_id']}: expected={row['expected']!r}  got={row['parsed']!r}  raw={row['raw_output']!r}  status={row['status']}")